# Estabilidade

Uma rede pode ter a arquitetura certa, a perda certa e o otimizador certo, e ainda assim treinar mal. O sinal que atravessa muitas camadas cresce ou encolhe a cada uma delas, e o gradiente que volta faz o mesmo, de modo que as primeiras camadas podem receber quase nada ou receber demais.

Este material trata das escolhas que mantêm esse fluxo sob controle: normalizar a entrada e as ativações, escolher a inicialização dos pesos, ajustar a taxa de aprendizado ao longo do treinamento e limitar gradientes grandes demais.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

In [ ]:
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Normalização da entrada

A primeira escala a controlar é a dos dados. Atributos em faixas muito diferentes produzem gradientes de magnitudes muito diferentes, e o mesmo passo do otimizador fica grande demais para uns e pequeno demais para outros. A correção é padronizar cada entrada,

$$
x' = \frac{x - \mu}{\sigma}
$$

em que $\mu$ e $\sigma$ são a média e o desvio padrão calculados sobre o conjunto de treino. O resultado tem média zero e desvio padrão um.

Os valores 0.1307 e 0.3081, usados nos materiais anteriores sem justificativa, são exatamente essas duas estatísticas para o MNIST.

In [ ]:
raw_train_set = datasets.MNIST(root="data", train=True, download=True, transform=transforms.ToTensor())
raw_images = next(iter(DataLoader(raw_train_set, batch_size=len(raw_train_set))))[0]

print(f"formato: {tuple(raw_images.shape)}")
print(f"média: {raw_images.mean().item():.4f}, desvio padrão: {raw_images.std().item():.4f}")

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
])

full_train_set = datasets.MNIST(root="data", train=True, download=True, transform=transform)
full_test_set = datasets.MNIST(root="data", train=False, download=True, transform=transform)

In [ ]:
train_set = Subset(full_train_set, range(4000))
validation_set = Subset(full_test_set, range(1000))

train_dataloader = DataLoader(train_set, batch_size=64, shuffle=True)
validation_dataloader = DataLoader(validation_set, batch_size=500, shuffle=False)

images, labels = next(iter(train_dataloader))
images, labels = images.to(device), labels.to(device)
print(f"treino: {len(train_set)}, validação: {len(validation_set)}")

## Gradientes que somem

O material sobre redes neurais mencionou que a sigmoid satura e que isso atrapalha o treinamento. Aqui dá para ver por quê. Ao atravessar uma ativação, o gradiente é multiplicado pela derivada dela. A derivada da sigmoid vale no máximo $0.25$, no centro, e cai para perto de zero nas duas pontas.

Em uma rede com $n$ camadas, o gradiente que chega à primeira passou por $n$ dessas multiplicações. Com fatores menores que um, o produto encolhe exponencialmente com a profundidade, e é isso que se chama de gradiente que some.

In [ ]:
z = torch.linspace(-5, 5, 200)

plt.figure(figsize=(8, 5))
plt.plot(z, torch.sigmoid(z) * (1 - torch.sigmoid(z)), label="sigmoid")
plt.plot(z, 1 - torch.tanh(z) ** 2, label="tanh")
plt.plot(z, (z > 0).float(), label="ReLU")
plt.xlabel("z")
plt.ylabel("derivada da ativação")
plt.legend()
plt.grid(True)
plt.show()

A tanh chega a derivada 1 no centro, mas também satura. A ReLU tem derivada exatamente 1 em todo o lado positivo, e é por isso que ela viabiliza redes profundas.

A rede abaixo tem oito camadas ocultas e é construída duas vezes, com sigmoid e com ReLU. Depois de um único `backward`, a norma do gradiente de cada camada mostra o que sobrou do sinal ao chegar no começo da rede.

In [ ]:
def deep_mlp(activation, depth=8, width=64):
    layers = [nn.Flatten(), nn.Linear(28 * 28, width), activation()]
    for _ in range(depth - 1):
        layers.append(nn.Linear(width, width))
        layers.append(activation())

    layers.append(nn.Linear(width, 10))
    return nn.Sequential(*layers).to(device)

In [ ]:
def gradient_norms(model):
    loss = nn.CrossEntropyLoss()(model(images), labels)

    model.zero_grad()
    loss.backward()

    return [layer.weight.grad.norm().item() for layer in model if isinstance(layer, nn.Linear)]

In [ ]:
torch.manual_seed(0)
sigmoid_norms = gradient_norms(deep_mlp(nn.Sigmoid))

torch.manual_seed(0)
relu_norms = gradient_norms(deep_mlp(nn.ReLU))

plt.figure(figsize=(8, 5))
plt.plot(sigmoid_norms, marker="o", label="sigmoid")
plt.plot(relu_norms, marker="o", label="ReLU")
plt.yscale("log")
plt.xlabel("camada")
plt.ylabel("norma do gradiente")
plt.legend()
plt.grid(True)
plt.show()

Com sigmoid, a norma do gradiente cai várias ordens de grandeza entre a última camada e a primeira. As camadas iniciais praticamente não são atualizadas, e como são elas que constroem as primeiras representações, a rede inteira aprende devagar. Com ReLU a queda é muito menor.

A escolha da ativação é a primeira defesa. As próximas duas seções tratam das outras duas, a inicialização e a normalização das ativações.

## Inicialização dos pesos

Antes da primeira iteração, os pesos precisam de algum valor, e essa escolha decide a escala do sinal que atravessa a rede.

Inicializar tudo com zero não funciona, e o motivo não é a escala. Todas as unidades de uma camada passam a calcular a mesma coisa, recebem o mesmo gradiente e continuam iguais para sempre. Nenhuma quantidade de treinamento quebra essa simetria, e a camada inteira se comporta como uma única unidade.

In [ ]:
symmetric = nn.Sequential(nn.Linear(4, 3), nn.ReLU(), nn.Linear(3, 1)).to(device)

with torch.no_grad():
    for layer in symmetric:
        if isinstance(layer, nn.Linear):
            nn.init.zeros_(layer.weight)
            nn.init.zeros_(layer.bias)

nn.MSELoss()(symmetric(torch.randn(8, 4, device=device)), torch.randn(8, 1, device=device)).backward()
print(symmetric[0].weight.grad)

As três linhas do gradiente são idênticas, e as três unidades vão receber a mesma atualização.

A saída é quebrar a simetria com valores aleatórios, mas a escala deles importa. Se forem grandes demais, o sinal cresce a cada camada e o gradiente explode; se forem pequenos demais, ambos encolhem. As duas receitas usuais escolhem a variância de modo a manter essa escala estável ao longo da profundidade,

$$
\text{Xavier:} \quad \mathrm{Var}[w] = \frac{2}{n_{in} + n_{out}}
\qquad
\text{He:} \quad \mathrm{Var}[w] = \frac{2}{n_{in}}
$$

em que $n_{in}$ e $n_{out}$ são o número de entradas e de saídas da camada. A de Xavier considera os dois sentidos do fluxo e serve para ativações simétricas como a tanh. A de He dobra a variância para compensar o fato de a ReLU zerar metade das ativações, e é o padrão em redes modernas.

In [ ]:
def apply_init(model, initializer):
    for layer in model:
        if isinstance(layer, nn.Linear):
            initializer(layer.weight)
            nn.init.zeros_(layer.bias)


def activation_stds(model):
    stds = []
    x = images

    with torch.no_grad():
        for layer in model:
            x = layer(x)
            if isinstance(layer, nn.Linear):
                stds.append(x.std().item())

    return stds

In [ ]:
initializers = {
    "normal com desvio 0.05": lambda w: nn.init.normal_(w, mean=0.0, std=0.05),
    "Xavier": nn.init.xavier_uniform_,
    "He": lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu"),
}

plt.figure(figsize=(8, 5))
for name, initializer in initializers.items():
    model = deep_mlp(nn.ReLU)
    apply_init(model, initializer)
    plt.plot(activation_stds(model), marker="o", label=name)

plt.yscale("log")
plt.xlabel("camada")
plt.ylabel("desvio padrão das ativações")
plt.legend()
plt.grid(True)
plt.show()

A inicialização normal com desvio fixo em 0.05 não leva em conta o tamanho da camada, e o sinal encolhe a cada uma delas até desaparecer. As duas receitas mantêm a escala aproximadamente constante ao longo da profundidade, que é exatamente o que elas foram desenhadas para fazer. O `nn.Linear` já usa uma inicialização desse tipo por padrão, e trocá-la só é necessário quando a rede foge do caso comum.

## Batch normalization

Mesmo com a entrada normalizada e os pesos bem inicializados, a escala das ativações internas muda enquanto os pesos são atualizados. A batch normalization ataca isso normalizando cada ativação dentro do mini lote,

$$
\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}
\qquad
y_i = \gamma \hat{x}_i + \beta
$$

em que $\mu_B$ e $\sigma_B^2$ são a média e a variância do lote, $\epsilon$ evita divisão por zero, e $\gamma$ e $\beta$ são parâmetros treináveis que devolvem à rede a liberdade de escolher a escala e o deslocamento. A camada estabiliza o treinamento e costuma permitir taxas de aprendizado maiores.

In [ ]:
batch_norm = nn.BatchNorm1d(128)
activations = torch.randn(32, 128) * 5 + 10

normalized = batch_norm(activations)

print(f"antes: média {activations.mean():.4f}, desvio padrão {activations.std():.4f}")
print(f"depois: média {normalized.mean():.4f}, desvio padrão {normalized.std():.4f}")

A camada tem comportamentos diferentes no treino e na avaliação. Durante o treino ela usa as estatísticas do lote atual e vai acumulando uma média móvel delas; na avaliação, usa a média acumulada, para que a previsão de um exemplo não dependa dos outros exemplos do lote. É o `model.train()` e o `model.eval()` que alternam entre os dois modos, e aqui essa distinção deixa de ser detalhe e passa a mudar o resultado.

## Decaimento da taxa de aprendizado

Um passo grande ajuda no começo, quando os parâmetros estão longe de qualquer mínimo, e atrapalha no fim, quando falta apenas ajustar. Um agendador, ou scheduler, reduz a taxa ao longo do treinamento. Os três mais comuns são

$$
\text{StepLR:} \quad \eta_t = \eta_0 \, \gamma^{\lfloor t / s \rfloor}
\qquad
\text{ExponentialLR:} \quad \eta_t = \eta_0 \, \gamma^{t}
$$

e o `ReduceLROnPlateau`, que não segue fórmula fixa: ele observa uma métrica de validação e multiplica a taxa por $\gamma$ quando ela para de melhorar por um número de épocas.

O agendador acompanha o otimizador, e seu `step` é chamado uma vez por época, depois do laço dos lotes.

In [ ]:
def schedule_curve(scheduler_class, epochs=30, **kwargs):
    parameter = torch.zeros(1, requires_grad=True)
    optimizer = torch.optim.SGD([parameter], lr=0.5)
    scheduler = scheduler_class(optimizer, **kwargs)

    rates = []
    for _ in range(epochs):
        rates.append(optimizer.param_groups[0]["lr"])
        optimizer.step()
        scheduler.step()

    return rates

In [ ]:
step_rates = schedule_curve(torch.optim.lr_scheduler.StepLR, step_size=10, gamma=0.5)
exponential_rates = schedule_curve(torch.optim.lr_scheduler.ExponentialLR, gamma=0.9)

plt.figure(figsize=(8, 5))
plt.plot(step_rates, label="StepLR, passo 10 e gama 0.5")
plt.plot(exponential_rates, label="ExponentialLR, gama 0.9")
plt.xlabel("época")
plt.ylabel("taxa de aprendizado")
plt.legend()
plt.grid(True)
plt.show()

## Gradient clipping

Um único lote atípico pode produzir um gradiente enorme e jogar os parâmetros para longe em um só passo, desfazendo o progresso de muitas épocas. O gradient clipping limita a norma do vetor de gradientes antes da atualização,

$$
g \leftarrow g \cdot \min\left(1, \frac{\tau}{\|g\|}\right)
$$

em que $\|g\|$ é a norma de todos os gradientes juntos e $\tau$ é o limite escolhido. Se a norma já é menor que $\tau$, nada muda; se é maior, o vetor é reescalado, preservando a direção e cortando só o tamanho.

In [ ]:
model = deep_mlp(nn.ReLU)
nn.CrossEntropyLoss()(model(images), labels).backward()

max_norm = 0.05
norm_before = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
norm_after = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)

print(f"norma antes do corte: {norm_before:.4f}")
print(f"norma depois do corte: {norm_after:.4f}")

O limite de 0.05 foi escolhido abaixo da norma observada, só para que o corte aconteça e seja visível. A função `clip_grad_norm_` devolve a norma que encontrou antes de cortar e altera os gradientes no lugar, e por isso a segunda chamada já devolve o valor cortado. No laço de treinamento ela entra entre o `backward` e o `step`, que é o único intervalo em que os gradientes já existem e ainda não foram usados.

## Treinamento completo

A rede abaixo junta as peças deste material. A batch normalization estabiliza as ativações internas, a inicialização de He acompanha a ReLU, o agendador reduz a taxa de aprendizado a cada cinco épocas e o clipping protege contra lotes atípicos.

In [ ]:
class StableMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )
        apply_init(self.layers, lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu"))

    def forward(self, x):
        return self.layers(x)   # [batch, 10]

In [ ]:
model = StableMLP().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
def evaluate(model, dataloader):
    model.eval()
    correct = 0

    with torch.no_grad():
        for batch_images, batch_labels in dataloader:
            batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
            correct += (model(batch_images).argmax(dim=1) == batch_labels).sum().item()

    return correct / len(dataloader.dataset)

In [ ]:
epochs = 15
train_losses = []
validation_accuracies = []
learning_rates = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for batch_images, batch_labels in train_dataloader:
        batch_images, batch_labels = batch_images.to(device), batch_labels.to(device)
        loss = criterion(model(batch_images), batch_labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * batch_images.size(0)

    learning_rates.append(optimizer.param_groups[0]["lr"])
    scheduler.step()

    train_losses.append(running_loss / len(train_set))
    validation_accuracies.append(evaluate(model, validation_dataloader))
    print(f"época {epoch + 1}: perda {train_losses[-1]:.4f}, "
          f"acurácia {validation_accuracies[-1]:.4f}, taxa {learning_rates[-1]:.5f}")

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))

ax1.plot(train_losses)
ax1.set_xlabel("época")
ax1.set_ylabel("entropia cruzada de treino")
ax1.grid(True)

ax2.plot(validation_accuracies)
ax2.set_xlabel("época")
ax2.set_ylabel("acurácia de validação")
ax2.grid(True)

ax3.plot(learning_rates)
ax3.set_xlabel("época")
ax3.set_ylabel("taxa de aprendizado")
ax3.grid(True)
plt.show()

A perda de treino cai continuamente até perto de zero, e as reduções da taxa de aprendizado tornam essa queda mais suave no fim. A acurácia de validação, no entanto, estaciona bem antes disso e não se move mais.

As duas curvas juntas dizem que, a partir de certo ponto, o que a rede ainda aprende vale apenas para os exemplos de treino. Nenhuma das técnicas deste material corrige isso, porque todas tratam do fluxo de sinal, e não da diferença entre decorar e generalizar. Esse é o assunto do próximo material.

## Exercícios

### Exercício 1

Repita o experimento das normas dos gradientes com uma rede de 16 camadas, e depois com uma de 4. Como a profundidade muda a distância entre as curvas da sigmoid e da ReLU?

In [ ]:
depths = [4, 8, 16]

### Exercício 2

Treine a `StableMLP` sem a batch normalization e sem o agendador, mantendo o resto igual, e compare as curvas com as obtidas acima. Qual das duas peças fez mais diferença aqui?

In [ ]:
# Monte a versão sem BatchNorm1d e treine com o mesmo laço.

### Exercício 3

Troque o `StepLR` pelo `ReduceLROnPlateau`, que precisa receber a métrica observada na chamada de `step`. Em que época ele decidiu reduzir a taxa, e como isso se compara com a redução fixa a cada cinco épocas?

In [ ]:
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, ...)